# AI Search Lab — Multilingual Retrieval Benchmark V1

**An exploratory Information Retrieval study for multilingual SEO queries.**  
**Author:** Federico Hauer · **Status:** Educational / exploratory · **Dataset:** Synthetic

### Research question
How do lexical retrieval (TF-IDF), semantic embeddings (MiniLM), and rank fusion (RRF) compare when ranking candidate page titles for multilingual search queries?

This notebook will:
1. Set up dependencies and load an open model from Hugging Face.
2. Define a small dataset and binary relevance judgments (*ground truth*).
3. Compute comparable rankings for three retrieval approaches.
4. Evaluate `Precision@3`, `Recall@3`, `Hit@1`, and `MRR@3`.
5. Save results and explain what **cannot** be inferred.

> **Scope:** 6 synthetic queries and 11 synthetic titles. This is not a study of Google's ranking systems, ChatGPT citations, or real search-engine performance. **Queries and document titles remain in their original German/Spanish languages intentionally.**


## 1. Environment and model

**Google Colab** provides a temporary Python runtime. **Sentence Transformers** is a Python library for loading models that produce vector representations (*embeddings*) of text.

Run the installation cell once in each fresh Colab runtime. We will download model files from Hugging Face and execute them on Colab CPU, **without using paid Hugging Face Inference Providers**. Free Colab resources are subject to availability and usage limits.


In [ ]:
!pip -q install sentence-transformers scikit-learn


In [ ]:
# Load the model and record library versions for future reproducibility checks.
import sys
from importlib.metadata import version
from sentence_transformers import SentenceTransformer

MODEL_ID = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_ID, device="cpu")

print("Loaded model:", MODEL_ID)
print("Embedding dimensions:", model.get_embedding_dimension())
print("Python:", sys.version.split()[0])
for package in ("sentence-transformers", "scikit-learn", "numpy", "torch", "transformers"):
    print(f"{package}: {version(package)}")


## 2. Dataset and relevance judgments

- A **document** here is a fabricated *page title*, not a full web page.
- A **query** represents a hypothetical user information need.
- **Ground truth** is the set of documents manually marked as relevant **before** ranking them.

The benchmark includes **6 queries × 11 documents = 66 query–document pairs**. Relevance is defined primarily by matching the requested service/action and destination. These binary labels are teaching assumptions, **not** independently validated judgments.

The data is embedded below so this notebook runs on its own. The repository also includes `data/benchmark_v1.json` for reuse outside Colab.


In [ ]:
# Synthetic page titles. Keep the original German/Spanish texts unchanged.
pages = {'P01': 'Skiverleih Winterberg | Ski ausleihen und online reservieren',
 'P02': 'Alquiler de esquís en Winterberg | Reserva tu equipo',
 'P03': 'Ski kaufen Winterberg | Neue Ski im Fachgeschäft',
 'P04': 'Skishop Winterberg | Ski kaufen und im Laden abholen',
 'P05': 'Skiservice Winterberg | Ski reparieren und Kanten schleifen',
 'P06': 'Hotel Winterberg | Zimmer und Apartments buchen',
 'P07': 'Snowboardverleih Winterberg | Snowboards mieten',
 'P08': 'Skiverleih Mayrhofen | Ski online mieten',
 'P09': 'Skiverleih Zell am See | Ausrüstung mieten',
 'P10': 'Skipass Winterberg | Liftkarten online buchen',
 'P11': 'Ski reparieren Innsbruck | Skiservice'}

# Relevance judgments: document IDs considered relevant for each query.
tests = [{'id': 'Q01', 'query': 'Wo kann ich in Winterberg Ski ausleihen?', 'relevant_ids': {'P02', 'P01'}},
 {'id': 'Q02', 'query': 'Wo kann ich in Winterberg Ski kaufen?', 'relevant_ids': {'P04', 'P03'}},
 {'id': 'Q03', 'query': 'Skireparatur in Winterberg', 'relevant_ids': {'P05'}},
 {'id': 'Q04', 'query': '¿Dónde alquilar esquís en Winterberg?', 'relevant_ids': {'P02', 'P01'}},
 {'id': 'Q05', 'query': 'Ski mieten in Mayrhofen', 'relevant_ids': {'P08'}},
 {'id': 'Q06', 'query': 'Hotel buchen in Winterberg', 'relevant_ids': {'P06'}}]

print("AI SEARCH LAB — BENCHMARK V1")
print("Documents:", len(pages))
print("Queries:", len(tests))
print("Query–document pairs:", len(pages) * len(tests))
assert len(pages) == 11 and len(tests) == 6
assert all(test["relevant_ids"] <= set(pages) for test in tests)

for test in tests:
    print(f"\n{test['id']}: {test['query']}")
    print("Relevant document IDs:", ", ".join(sorted(test["relevant_ids"])))


## 3. Lexical retrieval (TF-IDF) vs semantic retrieval (MiniLM)

**TF-IDF:** describes documents by weighted word features, including individual tokens and adjacent token pairs (*unigrams* and *bigrams*). Our basic setup cannot directly translate between languages and is not tuned for German compound words.

**MiniLM:** maps queries and titles to learned, dense embeddings. Cross-language and synonym relationships may be reflected, but **cosine similarity is not the same thing as intent equivalence, relevance probability, or an SEO ranking signal**.

Normalized MiniLM embeddings allow us to compute cosine similarity using a matrix product. TF-IDF cosine similarity is computed separately; we do **not** assume their numeric scores are directly calibrated to one another.


In [ ]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Preserve one document order for both methods and all evaluation steps.
document_ids = list(pages)
document_texts = [pages[doc_id] for doc_id in document_ids]
query_texts = [test["query"] for test in tests]

# Lexical baseline: word unigrams and bigrams.
vectorizer = TfidfVectorizer(lowercase=True, ngram_range=(1, 2))
tfidf_documents = vectorizer.fit_transform(document_texts)
tfidf_queries = vectorizer.transform(query_texts)
scores_tfidf = cosine_similarity(tfidf_queries, tfidf_documents)

# Semantic model: multilingual sentence embeddings.
document_embeddings = model.encode(document_texts, normalize_embeddings=True)
query_embeddings = model.encode(query_texts, normalize_embeddings=True)
scores_minilm = query_embeddings @ document_embeddings.T

assert scores_tfidf.shape == scores_minilm.shape == (6, 11)
print("TF-IDF score matrix:", scores_tfidf.shape)
print("MiniLM score matrix:", scores_minilm.shape)


## 4. Rank fusion with Reciprocal Rank Fusion (RRF)

**RRF** combines *positions in ranked lists*, rather than trying to add uncalibrated lexical and semantic similarity scores.

For each document, the score is the sum of `1 / (k + rank)` across both systems. We use **k = 60** and equal contribution from each list.

Ties (notably TF-IDF zero-score ties) are resolved consistently by **document ID**. This makes the example deterministic, but tied items do not necessarily carry meaningful relevance distinctions. **Hybrid retrieval is a hypothesis to evaluate, not a guaranteed improvement.**


In [ ]:
RRF_K = 60

def sorted_document_indices(score_row):
    """Return descending-score document indices; break ties by document ID."""
    return sorted(
        range(len(document_ids)),
        key=lambda j: (-float(score_row[j]), document_ids[j])
    )

scores_rrf = np.zeros_like(scores_minilm, dtype=float)
for query_index in range(len(tests)):
    tfidf_ranking = sorted_document_indices(scores_tfidf[query_index])
    minilm_ranking = sorted_document_indices(scores_minilm[query_index])
    for ranking in (tfidf_ranking, minilm_ranking):
        for rank, document_index in enumerate(ranking, start=1):
            scores_rrf[query_index, document_index] += 1.0 / (RRF_K + rank)

print("Hybrid RRF score matrix:", scores_rrf.shape)


## 5. Evaluation metrics

- **Precision@3:** fraction of the top three results labeled relevant.
- **Recall@3:** fraction of all labeled-relevant documents retrieved in the top three.
- **Hit@1:** whether the top-ranked result is labeled relevant.
- **MRR@3:** mean reciprocal rank of the **first relevant result** in the top three; 0 when none is found.

**Interpretation caveat:** If only one document in the entire collection is labeled relevant, `Precision@3` cannot exceed `1/3`, even with a perfect ranking. We therefore report multiple metrics and inspect per-query errors.


In [ ]:
# Evaluate all three systems against the SAME pre-defined relevance labels.
def evaluate_system(system_name, scores, k=3):
    records = []
    for query_index, test in enumerate(tests):
        top_indices = sorted_document_indices(scores[query_index])[:k]
        top_ids = [document_ids[j] for j in top_indices]
        relevant_ids = test["relevant_ids"]
        hits = sum(doc_id in relevant_ids for doc_id in top_ids)
        first_relevant_rank = next(
            (rank for rank, doc_id in enumerate(top_ids, start=1)
             if doc_id in relevant_ids),
            None
        )
        records.append({
            "system": system_name,
            "query_id": test["id"],
            "top_3": " | ".join(top_ids),
            "precision_at_3": hits / k,
            "recall_at_3": hits / len(relevant_ids),
            "hit_at_1": int(top_ids[0] in relevant_ids),
            "reciprocal_rank_at_3": (1.0 / first_relevant_rank)
                                     if first_relevant_rank is not None else 0.0,
        })
    return records

results = (
    evaluate_system("TF-IDF", scores_tfidf)
    + evaluate_system("MiniLM", scores_minilm)
    + evaluate_system("Hybrid RRF", scores_rrf)
)

print("SYSTEM        | P@3    | R@3    | Hit@1 | MRR@3")
print("-" * 61)
for system_name in ("TF-IDF", "MiniLM", "Hybrid RRF"):
    rows = [row for row in results if row["system"] == system_name]
    avg = lambda field: sum(row[field] for row in rows) / len(rows)
    print(f"{system_name:13} | {avg('precision_at_3'):.2%} | "
          f"{avg('recall_at_3'):.2%} | {avg('hit_at_1'):.2%} | "
          f"{avg('reciprocal_rank_at_3'):.2%}")

print("\nCASE STUDY: Q03 — ski repair query")
for row in results:
    if row["query_id"] == "Q03":
        print(f"{row['system']:13} -> {row['top_3']}")


## 6. Export results (optional)

Save per-query results to a CSV file. **Colab saves this file in its temporary runtime**, not automatically in your Google Drive. Download the file from Colab's Files panel if you want to retain it.

This export contains computed metrics from *the current run*. Historical results from the original learning session are preserved separately in the repository's `results/` directory.


In [ ]:
import csv

output_file = "benchmark_v1_results.csv"
with open(output_file, "w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=list(results[0]))
    writer.writeheader()
    writer.writerows(results)

print("Saved per-query results to:", output_file)


## 7. Findings and limitations

**Observed results from the original Colab experiment (2026-10-20):**

| System | Macro Recall@3 | Mean Hit@1 | MRR@3 |
|---|---:|---:|---:|
| TF-IDF | 66.67% | 83.33% | 83.33% |
| MiniLM | 83.33% | 83.33% | 88.89% |
| Hybrid RRF | 83.33% | 83.33% | 91.67% |

**Main observation:** On query Q03 (`Skireparatur in Winterberg`), relevant document P05 ranked outside the top three with TF-IDF, third with MiniLM, and second with Hybrid RRF. This **single query accounts for the observed MRR@3 difference**. Neither MiniLM nor the hybrid system retrieved P05 at rank one. The two rental queries also failed to retrieve both labeled-relevant cross-language titles within the top three.

**Important limitations:** small synthetic dataset; manually assigned binary relevance; no held-out test set; comparison of titles instead of actual page content; basic TF-IDF baseline; deterministic but arbitrary tie-breaking; RRF settings not tuned on a separate development set. Findings must **not** be generalized to real SEO rankings, AI Overviews, or the internal retrieval of commercial chatbots.

**Reproducibility:** The original notebook's exact dependency versions were not saved. The code above reports current versions, but stronger reproduction also requires frozen package versions and model revision. Running it again in Colab is a useful check, not proof of environment-independent reproducibility.

**Next phase — V2:** review relevance judgments, expand the dataset with appropriate permissions, create separate development/test splits, and compare stronger lexical baselines, retrieval-oriented embedding models, and rerankers. Avoid repeatedly tuning systems against the final test split.
